In [2]:
import numpy as np
import torch
import glob
from pathlib import Path
import pandas as pd
from scipy.spatial.distance import cdist

In [3]:
root = Path("/home/moritz.burmester/score-based-riemannian-metrics/experiments_urc")
pt_path  = root / "pt_files_urc"                       
latents = np.load(root / "dataset/latents.npy").astype(np.float32)   
n_angles = 360
 
print("real latents:", latents.shape)
print("pt folders:", sorted(p.name for p in pt_path.iterdir() if p.is_dir()))

real latents: (2520, 64)
pt folders: ['ebm', 'ebm_noise', 'rbf_land', 'rbf_land_noise', 'score_graph', 'score_graph_const', 'score_graph_const_noise', 'score_graph_noise', 'score_lerp_slerp']


In [7]:
## eval measures
def gamma_star(letters, a0, steps, N):
    # ideal uniform-rotation 
    us  = np.linspace(0, 1, N)
    idx = np.round(a0[:, None] + steps[:, None] * us[None, :]).astype(int) % n_angles
    return latents[letters[:, None] * n_angles + idx]
 
def d_and_gamma(paths, letters, a0, steps):
    # returns D-RMSE and gamma*-RMSE 
    P, N, D = paths.shape
    gs = gamma_star(letters, a0, steps, N)
    nn = cdist(paths.reshape(-1, D), latents).min(1).reshape(P, N)   # dist to manifold
    gd = np.linalg.norm(paths - gs, axis=-1)                      # dist to ideal rotation
    d_int = np.sqrt((nn[:, 1:-1] ** 2).mean(1))                   # interior points only
    g_int = np.sqrt((gd[:, 1:-1] ** 2).mean(1))
    return d_int, g_int
 
def agg(a):
    """mean and 2*SEM (approx 95% CI half-width)."""
    a = np.asarray(a)
    return float(a.mean()), float(2 * a.std(ddof=1) / np.sqrt(len(a)))

In [ ]:
def load_geo(folder, cfg):
    f = pt_path / folder / f"geo_{cfg}.pt"
    if not f.exists():
        return None
    d = torch.load(f, map_location="cpu", weights_only=False)
    p = d["paths"]; p = p.numpy() if torch.is_tensor(p) else np.asarray(p)
    L  = np.asarray(d["letters"]); a0 = np.asarray(d["a0"]); st = np.asarray(d["steps"])
    di, gi = d_and_gamma(p, L, a0, st)
    dm, de = agg(di); gm, ge = agg(gi)
    return dict(D=dm, D2sem=de, g=gm, g2sem=ge, n=p.shape[0])
 
def cellstr(r):
    if r is None: return "--"
    return f"{r['D']:.3f}±{r['D2sem']:.3f} | {r['g']:.3f}±{r['g2sem']:.3f}"

In [ ]:
lambdas   = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
lam_tag = {0.0:"lam00", 0.1:"lam01", 0.25:"lam025", 0.5:"lam05", 0.75:"lam075", 1.0:"lam10"}
 
# EBM (Etheta, invp) and baselines (RBF, LAND): files are "{name}_{init}".
#   0-noise folders: ebm, rbf_land        |  0.25-noise: ebm_noise, rbf_land_noise
# raw baselines "raw_lerp"/"raw_slerp" exist in every folder.
def ebm_dir(noise):  return "ebm_noise"      if noise else "ebm"
def base_dir(noise): return "rbf_land_noise" if noise else "rbf_land"
 
# diffusion score lambda-family location -> (folder, config):
#   lerp/slerp, 0 noise -> score_lerp_slerp / "{init}_lamXX"
#   graph, 0 noise      -> score_graph      / "lerp_lamXX"   (graph, historical label)
#   graph, 0.25 noise   -> score_graph_noise/ "graph_lamXX"
def score_loc(init, lam, noise):
    if noise:            return "score_graph_noise", f"graph_{LAMTAG[lam]}"
    if init == "graph":  return "score_graph",       f"lerp_{LAMTAG[lam]}"
    return "score_lerp_slerp", f"{init}_{LAMTAG[lam]}"
 
# constant-weight (arc-length) location:
#   graph, 0 noise    -> score_graph_const       / "lerp_lamXX"
#   graph, 0.25 noise -> score_graph_const_noise  / "graph_lamXX_cc"
def const_loc(lam, noise):
    if noise: return "score_graph_const_noise", f"graph_{LAMTAG[lam]}_cc"
    return "score_graph_const", f"lerp_{LAMTAG[lam]}"
 
def build_init_table(init, noise=False):
    ebm_dir  = FOLDERS["ebm_noise" if noise else "ebm"]
    base_dir = FOLDERS["rbf_land_noise" if noise else "rbf_land"]
    rows = []
    def add(label, r):
        rows.append(dict(metric=label,
                         D_RMSE=None if r is None else round(r["D"],3),
                         D_2sem=None if r is None else round(r["D2sem"],3),
                         gamma_RMSE=None if r is None else round(r["g"],3),
                         gamma_2sem=None if r is None else round(r["g2sem"],3),
                         n=None if r is None else r["n"]))
    add("raw lerp",  load_geo(ebm_dir, "raw_lerp"))
    add("raw slerp", load_geo(ebm_dir, "raw_slerp"))
    add("Etheta",    load_geo(ebm_dir, cfg_ebm("Etheta", init)))
    add("invp",      load_geo(ebm_dir, cfg_ebm("invp", init)))
    for lam in LAMS:
        if noise:
            r = try_load([cfg_score_noise(lam)])
        else:
            folder, cfg = cfg_score(init, lam); r = load_geo(folder, cfg)
        add(f"diff lam{lam}", r)
    add("RBF",  load_geo(base_dir, cfg_base("RBF", init)))
    add("LAND", load_geo(base_dir, cfg_base("LAND", init)))
    return pd.DataFrame(rows)

In [8]:

print("="*60, "\nTABLE 1a: lerp init, 0 noise\n", "="*60)
display(build_init_table("lerp"))
print("="*60, "\nTABLE 1b: slerp init, 0 noise\n", "="*60)
display(build_init_table("slerp"))
print("="*60, "\nTABLE 1c: graph init, 0 noise\n", "="*60)
display(build_init_table("graph"))

ebm                      ['Etheta_graph', 'Etheta_lerp', 'Etheta_slerp', 'invp_graph', 'invp_lerp', 'invp_slerp', 'raw_lerp', 'raw_slerp']
ebm_noise                ['Etheta_graph', 'Etheta_lerp', 'invp_graph', 'raw_lerp', 'raw_slerp']
rbf_land                 ['LAND_graph', 'LAND_lerp', 'LAND_slerp', 'RBF_graph', 'RBF_lerp', 'RBF_slerp', 'raw_lerp', 'raw_slerp']
rbf_land_noise           ['LAND_graph', 'RBF_graph', 'raw_lerp', 'raw_slerp']
score_graph              ['lerp_lam00', 'lerp_lam01', 'lerp_lam025', 'lerp_lam05', 'lerp_lam075', 'lerp_lam10', 'raw_lerp', 'raw_slerp']
score_graph_const        ['lerp_lam00', 'lerp_lam01', 'lerp_lam025', 'lerp_lam05', 'lerp_lam075', 'lerp_lam10', 'raw_lerp', 'raw_slerp']
score_graph_const_noise  ['graph_lam00_cc', 'graph_lam01_cc', 'graph_lam025_cc', 'graph_lam05_cc', 'graph_lam075_cc', 'graph_lam10_cc', 'raw_lerp', 'raw_slerp']
score_graph_noise        ['graph_lam00', 'graph_lam025', 'graph_lam05', 'graph_lam075', 'graph_lam10', 'lerp_lam00', 'lerp

,metric,D_RMSE,D_2sem,gamma_RMSE,gamma_2sem,n
0,raw lerp,2.055,0.290,2.338,0.349,100
1,raw slerp,1.340,0.299,1.489,0.341,100
2,Etheta,1.100,0.196,1.361,0.268,100
3,invp,1.855,0.286,2.327,0.412,100
4,diff lam0.0,2.074,0.402,2.545,0.495,100
5,diff lam0.1,1.171,0.254,1.493,0.323,100
6,diff lam0.25,1.173,0.258,1.531,0.325,100
7,diff lam0.5,1.219,0.267,1.735,0.343,100
8,diff lam0.75,1.349,0.286,2.002,0.375,100
9,diff lam1.0,1.853,0.337,2.641,0.458,100


TABLE 1b: slerp init, 0 noise


,metric,D_RMSE,D_2sem,gamma_RMSE,gamma_2sem,n
0,raw lerp,2.055,0.290,2.338,0.349,100
1,raw slerp,1.340,0.299,1.489,0.341,100
2,Etheta,0.890,0.145,1.365,0.266,100
3,invp,1.516,0.251,1.636,0.276,100
4,diff lam0.0,0.632,0.170,0.904,0.237,100
5,diff lam0.1,0.581,0.143,0.855,0.221,100
6,diff lam0.25,0.573,0.136,0.841,0.215,100
7,diff lam0.5,0.593,0.138,0.875,0.214,100
8,diff lam0.75,0.653,0.148,0.989,0.211,100
9,diff lam1.0,0.984,0.221,1.304,0.263,100


TABLE 1c: graph init, 0 noise


,metric,D_RMSE,D_2sem,gamma_RMSE,gamma_2sem,n
0,raw lerp,2.055,0.290,2.338,0.349,100
1,raw slerp,1.340,0.299,1.489,0.341,100
2,Etheta,0.744,0.083,1.532,0.479,100
3,invp,1.226,0.130,1.437,0.260,100
4,diff lam0.0,0.282,0.021,0.789,0.419,100
5,diff lam0.1,0.294,0.022,0.704,0.365,100
6,diff lam0.25,0.318,0.024,0.766,0.361,100
7,diff lam0.5,0.371,0.027,0.901,0.350,100
8,diff lam0.75,0.440,0.030,0.938,0.286,100
9,diff lam1.0,0.575,0.028,0.954,0.284,100


In [9]:
print("="*60, "\nTABLE 2: graph init, 0.25 endpoint noise\n", "="*60)
display(build_init_table("graph", noise=True))


TABLE 2: graph init, 0.25 endpoint noise


,metric,D_RMSE,D_2sem,gamma_RMSE,gamma_2sem,n
0,raw lerp,2.786,0.214,3.046,0.270,100
1,raw slerp,2.858,0.285,2.963,0.315,100
2,Etheta,1.372,0.066,2.195,0.452,100
3,invp,2.059,0.077,2.244,0.212,100
4,diff lam0.0,1.604,0.022,2.167,0.315,100
5,diff lam0.1,1.208,0.038,2.273,0.243,100
6,diff lam0.25,1.150,0.034,2.597,0.283,100
7,diff lam0.5,1.133,0.032,2.512,0.284,100
8,diff lam0.75,1.131,0.034,2.390,0.331,100
9,diff lam1.0,1.153,0.050,2.096,0.347,100


In [10]:
def build_score_const_table(noise=False):
    rows = []
    for lam in LAMS:
        if noise:
            s = try_load([cfg_score_noise(lam)])
            c = try_load(cfg_const_noise(lam))
        else:
            folder, cfg = cfg_score("graph", lam); s = load_geo(folder, cfg)
            c = try_load(cfg_const(lam))
        rows.append(dict(
            lam=lam,
            score_D=None if s is None else round(s["D"],3),
            score_D2=None if s is None else round(s["D2sem"],3),
            score_g=None if s is None else round(s["g"],3),
            const_D=None if c is None else round(c["D"],3),
            const_D2=None if c is None else round(c["D2sem"],3),
            const_g=None if c is None else round(c["g"],3),
        ))
    return pd.DataFrame(rows)
 
print("="*60, "\nTABLE 3: diffusion score vs constant, graph, 0 noise\n", "="*60)
display(build_score_const_table(noise=False))

TABLE 3: diffusion score vs constant, graph, 0 noise


,lam,score_D,score_D2,score_g,const_D,const_D2,const_g
0,0.00,0.282,0.021,0.789,0.282,0.021,0.789
1,0.10,0.294,0.022,0.704,0.277,0.020,0.674
2,0.25,0.318,0.024,0.766,0.276,0.020,0.666
3,0.50,0.371,0.027,0.901,0.299,0.023,0.675
4,0.75,0.440,0.030,0.938,0.405,0.041,0.664
5,1.00,0.575,0.028,0.954,1.938,0.263,2.190


In [11]:
print("="*60, "\nTABLE 4: diffusion score vs constant, graph, 0.25 noise\n", "="*60)
display(build_score_const_table(noise=True))

TABLE 4: diffusion score vs constant, graph, 0.25 noise


,lam,score_D,score_D2,score_g,const_D,const_D2,const_g
0,0.00,1.604,0.022,2.167,1.604,0.022,2.167
1,0.10,1.208,0.038,2.273,1.605,0.022,2.039
2,0.25,1.150,0.034,2.597,1.629,0.022,1.982
3,0.50,1.133,0.032,2.512,1.719,0.030,1.936
4,0.75,1.131,0.034,2.390,2.179,0.119,2.425
5,1.00,1.153,0.050,2.096,2.615,0.174,2.843
